# 🏠 End-to-End House Price Predictor
## 📊 Phase 1: Exploratory Data Analysis (EDA)

Welcome to Phase 1 of the End-to-End Data Science Project!
In this phase, we analyze the **Ames Housing Dataset** to uncover distributions, identify data quality issues, inspect correlations with our target variable (`SalePrice`), and detect anomalies or outliers before any machine learning modeling.

---

### 🎯 Key Objectives:
1. **Understand Structure**: Dimensionality, data types (numerical vs categorical).
2. **Data Hygiene**: Assess missing values and identify special encoding nuances.
3. **Target Analysis**: Inspect distribution of `SalePrice` and examine right-skewness.
4. **Feature Exploration**: Distribution of primary numerical and categorical predictors.
5. **Correlation & Relationships**: Uncover multi-collinearity and primary price drivers.
6. **Outlier Detection**: Spot unusual observations that could distort regression models.


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Suppress minor warnings for clean outputs
warnings.filterwarnings('ignore')

# Dark theme styling for clean visual aesthetics
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor": "#1a1d27",
    "axes.edgecolor": "#3a3d4d",
    "axes.labelcolor": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "text.color": "#c9d1d9",
    "grid.color": "#21262d",
    "grid.linewidth": 0.6,
    "font.family": "sans-serif",
    "font.size": 11,
})

ACCENT = "#58a6ff"   # Soft Blue
ACCENT2 = "#3fb950"  # Soft Green
ACCENT3 = "#ff7b72"  # Coral / Red

# Directory resolution (works whether running from notebooks/ or project root)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "raw" if (BASE_DIR / "data" / "raw").exists() else BASE_DIR / "datasets"
PLOTS_DIR = BASE_DIR / "plots" / "phase1"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Project Base: {BASE_DIR}")
print(f"📊 Data Source : {DATA_DIR}")
print(f"📈 Plots Output: {PLOTS_DIR}")


---
### 1. Data Loading & First Impressions
Let's load the training and test sets and verify their dimensions.


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(f"✔ Training set : {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"✔ Test set     : {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"ℹ Difference   : The test set has 1 fewer column ({set(train.columns) - set(test.columns)}), which is our prediction target.")

train.head(5)


---
### 2. Feature Type Breakdown & Descriptive Statistics
Separating numerical and categorical attributes helps structure the downstream cleaning and preprocessing pipelines.


In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()

print(f"🔢 Numerical features  : {len(num_cols)}")
print(f"🏷️ Categorical features: {len(cat_cols)}")

# Display numerical summary statistics (transposed for readability)
train[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].head(15)


---
### 3. Missing Value Analysis
Missing data in Ames Housing often has a specific domain meaning:
- In columns like `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`, `GarageType`, and `BsmtQual`, `NaN` does **not** mean missing at random; it denotes **absence of that feature** (e.g. No Pool, No Garage, No Basement).
- We will compute and visualize all features with missing entries.


In [ ]:
missing = (
    train.isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "Feature", 0: "MissingCount"})
)
missing["PercentMissing"] = (missing["MissingCount"] / len(train) * 100).round(2)
missing = missing[missing["MissingCount"] > 0].sort_values("PercentMissing", ascending=False)

print(f"Total features with missing values: {len(missing)}")
missing.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = [
    ACCENT3 if p > 50 else ACCENT if p > 15 else ACCENT2
    for p in missing["PercentMissing"]
]

bars = ax.barh(missing["Feature"], missing["PercentMissing"], color=colors, edgecolor="none")
ax.set_xlabel("% Missing", fontsize=12)
ax.set_title("Missing Values by Feature", fontsize=15, fontweight="bold", color="#e6edf3", pad=15)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

for bar, val in zip(bars, missing["PercentMissing"]):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%",
        va="center",
        fontsize=8,
        color="#8b949e"
    )

legend_elements = [
    Patch(facecolor=ACCENT3, label="> 50% missing (Rare feature or flag)"),
    Patch(facecolor=ACCENT,  label="15–50% missing (LotFrontage, FireplaceQu)"),
    Patch(facecolor=ACCENT2, label="< 15% missing (Garage, Basement features)"),
]
ax.legend(handles=legend_elements, loc="lower right", framealpha=0.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_missing_values.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 4. Target Variable Analysis (`SalePrice`)
Linear models assume normal residual errors and are sensitive to skewed response variables.
- We check the skewness of `SalePrice`.
- If skew > 1, a logarithmic transformation `log1p(x) = log(1 + x)` stabilizes variance and normalizes the target distribution.


In [ ]:
sp = train["SalePrice"]
skew_val = sp.skew()

print(f"Minimum Price : ${sp.min():,}")
print(f"Maximum Price : ${sp.max():,}")
print(f"Mean Price    : ${sp.mean():,.2f}")
print(f"Median Price  : ${sp.median():,.2f}")
print(f"Skewness      : {skew_val:.3f}  (Values > 1 indicate significant right-skew)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SalePrice: Raw vs Log-Transformed Distribution", fontsize=15, fontweight="bold", color="#e6edf3", y=1.01)

# Raw distribution
axes[0].hist(sp, bins=50, color=ACCENT, edgecolor="none", alpha=0.85)
axes[0].axvline(sp.mean(), color=ACCENT3, lw=2, linestyle="--", label=f"Mean: ${sp.mean()/1e3:.0f}k")
axes[0].axvline(sp.median(), color=ACCENT2, lw=2, linestyle="--", label=f"Median: ${sp.median()/1e3:.0f}k")
axes[0].set_title(f"Original SalePrice (Skew: {skew_val:.2f})", color="#e6edf3")
axes[0].set_xlabel("SalePrice ($)")
axes[0].yaxis.grid(True, alpha=0.3)
axes[0].legend()

# Log-transformed
log_sp = np.log1p(sp)
axes[1].hist(log_sp, bins=50, color=ACCENT2, edgecolor="none", alpha=0.85)
axes[1].axvline(log_sp.mean(), color=ACCENT3, lw=2, linestyle="--", label=f"Mean: {log_sp.mean():.2f}")
axes[1].set_title(f"log(1 + SalePrice) (Skew: {log_sp.skew():.2f} - Normal)", color="#e6edf3")
axes[1].set_xlabel("log(1 + SalePrice)")
axes[1].yaxis.grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_saleprice_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 5. Numerical Feature Distributions
Let's examine the distributions of key area, quality, and age indicators.


In [ ]:
key_num = [
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "GarageArea",
    "LotArea", "YearBuilt", "YearRemodAdd", "OverallQual",
    "OverallCond", "BedroomAbvGr"
]
key_num = [c for c in key_num if c in train.columns]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Key Numerical Feature Distributions & Skewness", fontsize=15, fontweight="bold", color="#e6edf3", y=1.02)
axes = axes.flatten()

for i, col in enumerate(key_num):
    data = train[col].dropna()
    axes[i].hist(data, bins=35, color=ACCENT, edgecolor="none", alpha=0.85)
    axes[i].set_title(col, color="#e6edf3", fontsize=11, fontweight="bold")
    axes[i].yaxis.grid(True, alpha=0.3)
    axes[i].set_xlabel(f"Skew: {data.skew():.2f}", fontsize=9, color="#8b949e")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_numerical_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 6. Correlation Analysis & Feature Multicollinearity
Pearson correlation measures linear relationships with `SalePrice`. We also check for collinearity between pairs of predictors (e.g. `GarageCars` & `GarageArea`, `TotalBsmtSF` & `1stFlrSF`).


In [ ]:
corr_matrix = train[num_cols].corr()
top_corr = (
    corr_matrix["SalePrice"]
    .drop("SalePrice")
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

print("Top 15 Absolute Correlations with SalePrice:")
for col, val in top_corr.items():
    print(f"  {col:<15} : {val:.3f}")

# Plot heatmap of Top 15 features + SalePrice
top15_cols = top_corr.index.tolist() + ["SalePrice"]
corr_sub = train[top15_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(
    corr_sub,
    mask=mask,
    cmap=cmap,
    center=0,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8},
    linewidths=0.5,
    linecolor="#0f1117",
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Correlation Heatmap — Top Predictors & Target", fontsize=14, fontweight="bold", color="#e6edf3", pad=15)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 7. Categorical Predictor Exploration
Neighborhood and zoning classifications often reflect geographic and socioeconomic pricing tiers.


In [ ]:
key_cat = ["Neighborhood", "MSZoning", "SaleCondition", "HouseStyle", "BldgType"]
key_cat = [c for c in key_cat if c in train.columns]

fig, axes = plt.subplots(len(key_cat), 1, figsize=(14, 5 * len(key_cat)))
fig.suptitle("Categorical Predictors vs. SalePrice", fontsize=15, fontweight="bold", color="#e6edf3")

for i, col in enumerate(key_cat):
    # Sort categories by median SalePrice
    order = train.groupby(col)["SalePrice"].median().sort_values(ascending=False).index
    sns.boxplot(
        data=train,
        x=col,
        y="SalePrice",
        order=order,
        palette="Blues_r",
        ax=axes[i],
        flierprops=dict(marker="o", markerfacecolor=ACCENT3, markersize=3, alpha=0.5)
    )
    axes[i].set_title(f"{col} vs SalePrice (Sorted by Median Price)", color="#e6edf3", fontsize=12)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=40)
    axes[i].yaxis.grid(True, alpha=0.3)
    axes[i].set_ylabel("SalePrice ($)")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_categorical_vs_price.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 8. Outlier Detection
The author of the Ames dataset (Dean De Cock) explicitly recommended removing homes with `GrLivArea > 4000 sq ft` that sold for unusually low prices (< $200k), as they represent unusual agricultural/partial sales.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Outlier Inspection: Living Area & Basement vs Price", fontsize=15, fontweight="bold", color="#e6edf3")

scatter_pairs = [("GrLivArea", "SalePrice"), ("TotalBsmtSF", "SalePrice")]

for ax, (x_col, y_col) in zip(axes, scatter_pairs):
    ax.scatter(train[x_col], train[y_col], alpha=0.45, s=25, color=ACCENT, edgecolors="none")
    ax.set_xlabel(f"{x_col} (sq ft)", fontsize=11)
    ax.set_ylabel("SalePrice ($)", fontsize=11)
    ax.set_title(f"{x_col} vs {y_col}", color="#e6edf3")
    ax.yaxis.grid(True, alpha=0.3)
    ax.xaxis.grid(True, alpha=0.3)
    
    # Flag suspect outliers
    if x_col == "GrLivArea":
        suspects = train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 200000)]
        ax.scatter(suspects[x_col], suspects[y_col], color=ACCENT3, s=70, zorder=5, label=f"Extreme Outliers ({len(suspects)})")
        ax.legend()
        print(f"Detected {len(suspects)} suspect outlier rows with IDs: {suspects['Id'].tolist()}")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_outlier_inspection.png", dpi=150, bbox_inches="tight")
plt.show()


---
### 9. Overall Quality vs Median Price
`OverallQual` is our single strongest predictor ($r = 0.791$). Let's inspect the pricing progression across quality tiers (1 to 10).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

quality_stats = train.groupby("OverallQual")["SalePrice"].median()
bar_colors = plt.cm.Blues(np.linspace(0.3, 0.95, len(quality_stats)))

bars = ax.bar(quality_stats.index, quality_stats.values, color=bar_colors, edgecolor="none")
ax.set_xlabel("Overall Quality Rating (1 = Very Poor → 10 = Very Excellent)", fontsize=11)
ax.set_ylabel("Median SalePrice ($)", fontsize=11)
ax.set_title("Overall Quality vs Median SalePrice (Exponential Tier Growth)", fontsize=14, fontweight="bold", color="#e6edf3", pad=15)
ax.yaxis.grid(True, alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 3000,
        f"${height/1e3:.0f}k",
        ha="center",
        fontsize=8.5,
        color="#8b949e"
    )

plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_quality_vs_price.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 🏁 Phase 1 Summary & Transition to Phase 2

### 📌 Core Takeaways:
1. **Target Distribution**: `SalePrice` has a positive skew of `1.88`. Applying $\log(1 + y)$ will normalize residuals and improve model stability.
2. **Missing Values**:
   - High missing columns (`PoolQC`, `MiscFeature`, `Alley`, `Fence`) are domain-specific "No Feature" flags rather than random omissions.
   - We will encode them systematically in Phase 2 rather than naively dropping features.
3. **Primary Predictive Signals**:
   - `OverallQual` ($r = 0.791$)
   - `GrLivArea` ($r = 0.709$)
   - `GarageCars` / `GarageArea` ($r \approx 0.63$)
   - `TotalBsmtSF` / `1stFlrSF` ($r \approx 0.61$)
4. **Outliers**:
   - Found 2 extreme outliers in `GrLivArea` (> 4000 sq ft, price < $200,000) that should be removed during cleaning to prevent skewing linear boundaries.

---
### 🚀 Ready for Phase 2: Data Cleaning & Preprocessing
Next, we will build:
- Outlier trimming
- Domain-aware missing value imputation (e.g. median by neighborhood for `LotFrontage`, `'None'` for missing amenities)
- Ordinal vs One-Hot categorical transformations
- Skewness remediation for numerical features
